# Treinamento Blindado v2.0 (QLoRA Fine-Tuning) — Especialista Acadêmico Luiz Gabriel (Llama 3.1 8B)
**O que mudou no Modelo 2.0 (Calibrado nos 5 Blocos Ouro 86%–99% Human do GPTZero 4.1m):**
1. **Dataset 100% Verificado no GPTZero 4.1m (`dataset_treino_v2.jsonl`, 188 exemplos):** Contém exclusivamente os 5 blocos aprovados com 86% a 99% Human (Cap. 2 = 99%, Seção 1.1 = 97%, Seções 1.2+1.3 = 94%, Seção 3.1 = 88% e Resumo = 86%), com alinhamento 1:1 estrito de parágrafos (`in_paras == out_paras`).
2. **Treino Exclusivo na Resposta (`DataCollatorForCompletionOnlyLM`):** O modelo calcula o gradiente **somente depois de `### Resposta:`**, ignorando os vícios sintáticos do texto de entrada.
3. **QLoRA Rank 32 (`r=32, lora_alpha=64`) + `neftune_noise_alpha=5` + Scheduler Cosseno:** Aprende a estrutura sintática sem decorar frases isoladas de outros capítulos.
4. **Blindado contra quedas de RAM/Wi-Fi:** Salva no Google Drive e limpa a VRAM/RAM antes de exportar o `.gguf`.

In [ ]:
# CÉLULA 1: Conectar Google Drive + Subir dataset_treino_v2.jsonl
import os, shutil
from google.colab import drive, files

drive.mount('/content/drive')

drive_dataset = "/content/drive/MyDrive/dataset_treino_v2.jsonl"
local_dataset = "dataset_treino_v2.jsonl"

if os.path.exists(local_dataset):
    shutil.copy(local_dataset, drive_dataset)
elif not os.path.exists(drive_dataset):
    print("Selecione o arquivo dataset_treino_v2.jsonl (ou dataset_treino_luiz.jsonl) no seu computador:")
    uploaded = files.upload()
    up_name = list(uploaded.keys())[0]
    shutil.copy(up_name, drive_dataset)

shutil.copy(drive_dataset, local_dataset)
print(f"✅ Google Drive conectado e {local_dataset} garantido no Drive!")

In [ ]:
# CÉLULA 2: Instalar Unsloth
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
# CÉLULA 3: Carregar Llama 3.1 8B + QLoRA (r=32, alpha=64) e preparar os 188 exemplos 100% verificados
from unsloth import FastLanguageModel
from datasets import load_dataset
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

alpaca_prompt = """Abaixo está uma instrução que descreve uma tarefa, acompanhada de uma entrada que fornece o contexto. Escreva uma resposta que conclua adequadamente a solicitação sem qualquer metatexto, mantendo exatamente o mesmo número de parágrafos e todos os fatos da entrada.

### Instrução:
{}

### Entrada:
{}

### Resposta:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    texts = [
        alpaca_prompt.format(inst, inp, out) + EOS_TOKEN
        for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"])
    ]
    return { "text" : texts }

dataset = load_dataset("json", data_files="dataset_treino_v2.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"✅ Modelo v2.0 e {len(dataset)} exemplos padrão-ouro carregados!")

In [ ]:
# CÉLULA 4: Treinar calculando a Loss SOMENTE na Resposta (mascarando a Entrada de IA!)
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from unsloth import is_bfloat16_supported
import torch

pasta_drive_lora = "/content/drive/MyDrive/lora_luiz_v2_treinado"

# Collator blindado (compatível com qualquer versão do TRL/Colab): mascara com -100 tudo antes de "### Resposta:\n"
class ResponseOnlyCollator(DataCollatorForLanguageModeling):
    def __init__(self, tokenizer):
        super().__init__(tokenizer=tokenizer, mlm=False)
        self.resp_ids = tokenizer.encode("### Resposta:\n", add_special_tokens=False)
    def torch_call(self, examples):
        batch = super().torch_call(examples)
        labels = batch["labels"]
        m = len(self.resp_ids)
        for i in range(labels.size(0)):
            seq = labels[i].tolist()
            for j in range(len(seq) - m + 1):
                if seq[j:j+m] == self.resp_ids:
                    labels[i, :j+m] = -100
                    break
        return batch

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = ResponseOnlyCollator(tokenizer),
    dataset_num_proc = 2,
    packing = False,
    neftune_noise_alpha = 5,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 8,
        num_train_epochs = 3,
        learning_rate = 1.5e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        save_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.03,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = pasta_drive_lora,
        report_to = "none",
    ),
)

trainer_stats = trainer.train()
model.save_pretrained(pasta_drive_lora)
tokenizer.save_pretrained(pasta_drive_lora)
print(f"✅ TREINO V2.0 SALVO DEFINITIVAMENTE NO GOOGLE DRIVE EM: {pasta_drive_lora}")

In [ ]:
# CÉLULA 5: Limpar memória RAM do Trainer e Exportar Modelo v2.0 para GGUF
import gc, torch, os, shutil

try:
    del trainer
    del dataset
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

model.save_pretrained_gguf(
    "modelo_luiz_v2_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
    maximum_memory_usage = 0.5,
)

for fname in os.listdir("modelo_luiz_v2_gguf"):
    if fname.endswith(".gguf"):
        src = os.path.join("modelo_luiz_v2_gguf", fname)
        dst = "/content/drive/MyDrive/luiz-academico-v2-q4_k_m.gguf"
        shutil.copy(src, dst)
        print(f"🎉 SUCESSO TOTAL! Seu modelo v2.0 GGUF está salvo no Google Drive em: {dst}")